In [28]:
load('IrreducibilityLeanProofWriter.sage')
load('RingOfIntegersLeanProofWriter.sage')
%run CertificateClassGroupLean2.0.ipynb

In [108]:
# Writes a Lean proof for the ring of integers of a number field, given an integral basis. 

# Input: 
# T : a monic irreducible polynomial defining the number field
# basis : an integral basis for the ring of integers of K, written as a list of polynomials (over a root of T)
# f : the file where the Lean code will be written. 
# nameIrr : the name of the file which contains a proof irreducible_T proving the irreducibility of T
# comment : comment in the beginning of the file 
# flagD : set to 1 if it includes proof of discriminant. 

def LeanProof(T, basis, f, nameIrr, num, comment, flagD):
    d = T.degree()
    C , denom = BasisToMatrix(basis)
    print(f"""
import IdealArithmetic.CertificateDedekind
import IdealArithmetic.CertifyAdjoinRoot
import Mathlib.Tactic.NormNum.Prime
import IdealArithmetic.MaximalAPI
import Mathlib.NumberTheory.NumberField.Basic
import IdealArithmetic.Examples.NF{num}.{nameIrr}""", file = f)
    if flagD == 1:
        print(f"""import IdealArithmetic.DiscriminantSubalgebraBuilder""", file = f)
    print(f"""
{comment}

open Polynomial

noncomputable def T : ℤ[X] := {T} 
lemma T_def : T = {T} := rfl

def K := AdjoinRoot (map (algebraMap ℤ ℚ) T)

noncomputable instance : CommRing K := by
  unfold K
  infer_instance

noncomputable instance : Algebra ℚ K := by
  unfold K
  infer_instance

local notation "l" => {T.list()}

noncomputable def Adj : IsAdjoinRoot K (map (algebraMap ℤ ℚ) T) :=
   AdjoinRoot.isAdjoinRoot _
   
local notation "θ" => Adj.root

lemma T_ofList : ofList l = T := by
  rw [T_def] ; norm_num ; ring
""", file = f)
    print(f"-- We build the subalgebra with integral basis {basis} ", file = f)
    print("", file = f)
    Table, auxTemp = SubalgebraBuilderF(T, C, denom, f)

    print(f"""
lemma T_degree : T.natDegree = {T.degree()} := (SubalgebraBuilderOfList T l BQ).hdeg

lemma T_monic : Monic T := by
  rw [← T_ofList]
  refine monic_ofList l rfl

lemma T_irreducible : Irreducible T := irreducible_T

noncomputable def Om : Subalgebra ℤ K := integralClosure ℤ K

noncomputable def O := subalgebraOfBuilderLists T l BQ

def hm : O ≤ Om := le_integralClosure_of_basis O (basisOfBuilderLists T l BQ)

noncomputable def B' : Basis (Fin {T.degree()}) ℤ Om :=
  Basis.reindex (AdjoinRoot.basisIntegralClosure T_monic
    (Irreducible.prime T_irreducible)) (finCongr T_degree)

instance OmFree : Module.Free ℤ Om := Module.Free.of_basis B'
instance OmFinite : Module.Finite ℤ Om := Module.Finite.of_basis B'

noncomputable def timesTableO : TimesTable (Fin {T.degree()}) ℤ O :=
  timesTableOfSubalgebraBuilderLists T l BQ 

noncomputable def B : Basis (Fin {T.degree()}) ℤ O := timesTableO.basis 
""", file = f)

    strTa = ""
    print(f'def Table : Fin {T.degree()} → Fin {T.degree()} → List ℤ := \n ![', end = "", file = f)
    for i in range(d):
        if i != d -1 :
            strTa = strTa + ' !' + str(Table[i]) + ', \n'
        else : 
            strTa = strTa + ' !' + str(Table[i]) 
    print(strTa + ']', file = f)

    print(f"""
lemma timesTableT_eq_Table :  ∀ i j , Table i j = List.ofFn (timesTableO.table i j) := by decide

lemma hroot_mem : θ ∈ O := by
  refine root_in_subalgebra_lists T l BQ !{auxTemp.list()} [] (by decide)
""", file=f)

    bad = CertificateDedekindF(T, f)
    flagl = []
    flaglW = []
    for p in bad:
        flagp , flagW = CertificatePMaximalityF(T, C, denom, p, Table, f)
        flagl = flagl + [(p, flagp)] 
        flaglW = flaglW + [(p, flagW)]
    flagl = dict(flagl)
    flaglW = dict(flaglW)
    print("", file = f)

    f.close()
    return bad, flagl, flaglW, flagD

In [201]:
R.<X>= PolynomialRing(ZZ)
K.<a>= PolynomialRing(QQ)
B = [1, a, a^2, 1/7*a^3 + 3/7*a^2 + 1/7*a + 2/7, 1/14*a^4 - 1/14*a^2 + 3/7*a - 3/7]
T = X^5 + 25*X^3 - 70*X^2 + 90*X - 88
num = '67'
f = open(f"RI{num}.lean", "w")
doc = open(f"Irreducible{num}.lean", "w")
bad, flagl, flaglW, flagD = LeanProof(T,B,f,f'Irreducible{num}',num, '',0)
S.<x>= PolynomialRing(ZZ)
R.<X>= PolynomialRing(ZZ)
LeanProofIrreducible(R(S(T)), doc)

In [202]:
import re
K.<a> = NumberField(T, 'a')
O = K.ring_of_integers()
Cl = K.class_group('pari')
n = K.degree()
a = K.gen()
B = [K(x) for x in B]
ideal_gens = Cl.gens()[0].ideal().gens_reduced()
J = O.ideal(ideal_gens)
m = Cl(J).order()
x = (J ^ m).gens_reduced()[0]
pr = m.prime_divisors()[0]
A , Q , elems , flag = MatrixPrimes (K, O, pr, x, 200)
elems = [K(r) for r in elems]
Ql = [list(Q[i].gens()) for i in range(len(Q))]
Qlq = [Ql[i][0] for i in range(len(Ql))]
if flag == 0 : 
    names = ['zeta' + str(i + 1) for i in range(len(Q) - 1)] + ['alpha']
else : 
    names = ['zeta' + str(i + 1) for i in range(len(Q) - 2)] + ['v'] + ['alpha']

In [203]:
%%capture capNP
print(f"""import IdealArithmetic.IdealArithmetic
import Mathlib.NumberTheory.NumberField.Units.DirichletTheorem
import IdealArithmetic.PrincipalityCertificate
import Mathlib.RingTheory.AdjoinRoot
import IdealArithmetic.Examples.NF{num}.RI{num}

open BigOperators Classical Matrix Polynomial

lemma B_one : B 0 = 1 := by
  refine basisOfBuilderLists_zero_eq_one _ _ BQ

lemma B_one_repr : B.equivFun.symm !{[1] + [0 for i in range(len(B) - 1)]} = 1 := by
  rw [Basis.equivFun_symm_eq_repr_symm']
  apply_fun B.repr
  rw [← B_one]
  simp only [Basis.repr_symm_apply, Basis.repr_linearCombination, Fin.isValue, Basis.repr_self]
  ext i
  fin_cases i <;> norm_num
  · exact LinearEquiv.injective B.repr """)

print("""
instance : IsDomain O := by
  haveI hirr : Fact $ Irreducible (map (algebraMap ℤ ℚ) T) :=
  {out := (Polynomial.Monic.irreducible_iff_irreducible_map_fraction_map (T_monic)).1 T_irreducible}
  letI hola : Field K := by
    unfold K
    exact AdjoinRoot.instField
  haveI : IsDomain K := by infer_instance
  refine Subalgebra.isDomain O

noncomputable section""")

L, M = DiscreteLogListLean(K, B, Ql, 'I', pr, elems, names)
IdealPowLean(B, 'J' + str(m), ideal_gens, 'J', x, 'alpha', m)
IsUnitInvLean(B, elems[:-1],names[:-1])
if flag == 1 :
    order_v = TorsionUnitProof(K, B, elems[len(elems) - 2], names[len(names) - 2])

In [204]:
%%capture capInv
print(f"""import IdealArithmetic.Examples.NF{num}.NonPrincipalExample{num}

/- Number field `K(α)` with with α root of polynomial `{T}`.
Class number `{m}`, generated by class of ideal `J = ({ideal_gens[0]}, {ideal_gens[1]})`. We have `J ^ {pr}` is principal. -/

/- Ring of integers with basis `{B}` -/

open BigOperators Classical Matrix Polynomial

noncomputable section

instance hirr : Fact $ (Irreducible (map (algebraMap ℤ ℚ) T)) where
  out :=  (Polynomial.Monic.irreducible_iff_irreducible_map_fraction_map (T_monic)).1 T_irreducible """)

print(f"""
noncomputable instance K_field : Field K := by
  unfold K
  infer_instance

instance K_numberField : NumberField K := by
  unfold K
  infer_instance

lemma K_finrank : Module.finrank ℚ K = {len(B)} := by
  unfold K
  rw [Module.finrank_eq_card_basis (AdjoinRoot.powerBasisAux _), Polynomial.natDegree_map_eq_of_injective, T_degree]
  · simp
  · exact RingHom.injective_int (algebraMap ℤ ℚ)
  · exact Irreducible.ne_zero hirr.out """)

print(f"""
theorem O_integral_closure : O = integralClosure ℤ K := by
  refine eq_of_piMaximal_at_all_primes_int O Om hm ?_
  intro p hp
  by_cases hc : p ∈ {bad}
  · fin_cases hc""")
for p in bad:
    if flagl[p] == 0:
        if flaglW[p] == 1:
            print(f'    exact pMaximal_of_MaximalOrderCertificateWLists {p} O Om hm M{p}')
        else : 
            print(f'    exact pMaximal_of_MaximalOrderCertificateLists {p} O Om hm M{p}')
    else : 
        print(f'    exact pMaximal_of_MaximalOrderCertificateOfUnramifiedLists {p} O Om hm M{p}')
print(f"""  · haveI : Fact $ Nat.Prime p := fact_iff.2 hp
    refine piMaximal_of_root_in_order_of_satisfiesDedekindCriterion_int Adj T_monic hm ?_ hroot_mem
     (satisfiesDedekindAlmostAllLists_of_certificate T _ T_ofList {bad} D p hp hc)
    rw [T_degree, rank_subalgebra_eq_card_basis Om B']
""")
print("")
print(f"""theorem  O_ringOfIntegers' : O = NumberField.RingOfIntegers K := by rw [O_integral_closure] ; rfl
""")
if flagD == 1:
    K.<b>=NumberField(T)
    D = K.discriminant()
    print(f"""
lemma T_discr : T.discriminant = {T.discriminant()} :=  by
  rw [T_monic.discriminant_def, T_degree, ← T_ofList]
  have : {list(T)}.derivative = {list(derivative(T)) + [0]} := rfl
  rw [← ofList_derivative_eq_derivative , this]
  decide

theorem K_discr : NumberField.discr K = {D} := by
  rw [discr_numberField_eq_discrSubalgebraBuilder
  T_irreducible BQ O_integral_closure]
  rw [T_discr]
  rfl

""")

print("""
instance : Module.Finite ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFiniteIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance : Module.Free ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFreeIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance :  Fintype ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFintypeSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

instance : IsCyclic ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instIsCyclicSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

""")
RankUnitCertificateLean(R(T), 'T_ofList', 'Adj', 'O_integral_closure', 'RC')
print('')
if flag == 0:
    pNeDvdTorsionLean(K, B, 'K_finrank' , 'O_integral_closure', 'IC', pr)
    print('')
    CertificateNonPrincipalityNT (L, pr, m, names[:-1], 'I', 'J', 'alpha', Qlq, 'RC', 'J' + str(m))
    print("""
theorem J_not_principal : ¬ ∃ b, J = Ideal.span {b} :=
  not_principal_of_NonPrincipalCertificateNDvdT NPCJ """)
else : 
    print('')
    CertificateNonPrincipalityDVDT (L, p, m, names[:-2], names[len(names) - 2], order_v, 'I', 'J', 'alpha', Qlq, 'RC', 'J' + str(m))
    print("""
theorem J_not_principal : ¬ ∃ b, J = Ideal.span {b} :=
  not_principal_of_NonPrincipalCertificateDvdT NPCJ """)

In [205]:
with open(f"NonPrincipalExample{num}.lean", "w") as f:
    f.write(capNP.stdout)

with open(f"Invariants{num}.lean", "w") as f:
    f.write(capInv.stdout)

In [236]:
R.<x> = PolynomialRing(ZZ)
f = R([-1,2,56,2,3,4,5,-7,1])
g = R([-18, -15, -10, 10, 0, 1])
h = x^12 - 2*x^11 + 2*x^10 - x^9 + 2*x^8 - 5*x^7 + 8*x^6 - 7*x^5 + 4*x^4 - 3*x^3 + 4*x^2 - 3*x - 1
j = x^7 - 2*x^6 + 4*x^5 - 4*x^4 + 3*x^3 - x^2 - x + 1
ten = x^10 - 3*x^9 + 7*x^8 - 11*x^7 + 13*x^6 - 12*x^5 + 9*x^4 - 5*x^3 + 3*x^2 - 2*x + 1
ten.discriminant()

-209352647

In [237]:
SturmCertificateLean(ten, 'T10')

def T10 : SturmBuilderOfList [[1, -2, 3, -5, 9, -12, 13, -11, 7, -3, 1], [-2, 6, -15, 36, -60, 78, -77, 56, -27, 10], [-94, 162, -195, 242, -360, 366, -289, 162, -59], [95, 302, -381, -168, 758, -690, 599, -288], [1931, -4378, 7359, -7080, 5294, -2778, 1571], [455, -26960, 47725, -47176, 25063, -13068], [-134179, 332692, -344321, 151604, -5075], [-1456762, 3563776, -3616063, 1519262], [35314526, -48265420, 29334551], [13943, 12602], [-1]] [1, -2, 3, -5, 9, -12, 13, -11, 7, -3, 1] [-2, 6, -15, 36, -60, 78, -77, 56, -27, 10] where
  hlen := by decide
  h0 := by decide
  h1 := by decide
  hlast := by decide
  hdrop := by decide
  hmono := by
    dsimp
    intro i hic 
    have hi : i < 10 := by omega
    interval_cases i <;> (dsimp ; decide)
  e := [100, 3481, 82944, 2468041, 170772624, 25755625, 2308157024644, 860515882371601, 158810404]
  f := [1, 100, 3481, 82944, 2468041, 170772624, 25755625, 6963709743350948, 19791865294546823]
  epos := by decide
  fpos := by decide
  Q := [[-3, 10]

[[1, -2, 3, -5, 9, -12, 13, -11, 7, -3, 1],
 [-2, 6, -15, 36, -60, 78, -77, 56, -27, 10],
 [-94, 162, -195, 242, -360, 366, -289, 162, -59],
 [95, 302, -381, -168, 758, -690, 599, -288],
 [1931, -4378, 7359, -7080, 5294, -2778, 1571],
 [455, -26960, 47725, -47176, 25063, -13068],
 [-134179, 332692, -344321, 151604, -5075],
 [-1456762, 3563776, -3616063, 1519262],
 [35314526, -48265420, 29334551],
 [13943, 12602],
 [-1]]

In [ ]:
R([)